In [2]:
#Step1
import pandas as pd
import pandasql as ps
import pixiedust
import sys
sys.path.append('..')
import util
import etl
import pyarrow.parquet as pq
import pyarrow as pa
!pip install duckdb
import duckdb
import inspect

# Get the path of the imported module (etl.py)
etl_module_path = inspect.getfile(etl)
print("Path of the imported ETL module (etl.py):", etl_module_path)
util.usedatabase(spark, "real_world_data_jun_2022")

pixiedust.enableJobMonitor()

con = duckdb.connect()

Please see https://github.com/pypa/pip/issues/5599 for advice on fixing the underlying issue.
To avoid this problem you can invoke Python with '-m pip' instead of running pip directly.
Path of the imported ETL module (etl.py): /home/o_suchsi/work/Oklahoma State/Priya/epilepsy/etl.py
Using real_world_data_jun_2022 ....
Spark Job Progress Monitor already enabled


In [3]:
#Step2
Epilepsy_Combined = spark.read.parquet("file:/home/o_suchsi/work/Oklahoma State/Priya/epilepsy/priya-smallset-finalF1_Numbered_NoNullStr.parquet")

In [4]:
#Step3
balanced_df = Epilepsy_Combined.drop("personid")

In [5]:
#Step4
from pyspark.sql import functions as F
from pyspark.sql import DataFrame
from pyspark.sql import SparkSession
import random
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, VectorSizeHint

# Step 0: Mimicking randomSplit with seed and randomness
def probabilistic_split(df: DataFrame, fractions: list, seed=None) -> list:
    if seed is not None:
        random.seed(seed)

    cumulative_fractions = [sum(fractions[:i + 1]) for i in range(len(fractions))]
    random_col = F.rand(seed)
    df_with_random = df.withColumn("random", random_col)
    splits = []
    prev_fraction = 0
    for fraction in cumulative_fractions:
        split_df = df_with_random.filter((F.col("random") >= prev_fraction) & (F.col("random") < fraction))
        splits.append(split_df.drop("random"))
        prev_fraction = fraction
    return splits

# Train, validation, and test sampling fractions
train_fraction = 0.7
valid_fraction = 0.2
test_fraction = 0.1
fractions = [train_fraction, valid_fraction, test_fraction]
seed_value = 23
train_data, valid_data, test_data = probabilistic_split(balanced_df, fractions, seed=seed_value)

# Show class distribution in train, validation, and test datasets
train_data.groupBy('label').count().show()
valid_data.groupBy('label').count().show()
test_data.groupBy('label').count().show()

# Print counts
print("Sampled data count:", balanced_df.count())
print("Train data count:", train_data.count())
print("Validation data count:", valid_data.count())
print("Test data count:", test_data.count())

+-----+-----+
|label|count|
+-----+-----+
|  0.0|70345|
|  1.0|10738|
+-----+-----+

+-----+-----+
|label|count|
+-----+-----+
|  0.0|20222|
|  1.0| 2975|
+-----+-----+

+-----+-----+
|label|count|
+-----+-----+
|  0.0|10227|
|  1.0| 1565|
+-----+-----+

Sampled data count: 116072
Train data count: 81083
Validation data count: 23197
Test data count: 11792


In [6]:
#Step5
columns = train_data.columns
feature_cols = columns
feature_cols.remove('label')
target_col = ['label']

In [7]:
#Step6
###########################Included Standard Scalar #############################################################
from pyspark.sql import DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, VectorSizeHint, StandardScaler

# Step 1: Define vector columns and non-vector columns based on schema
vector_cols = [col_name for col_name, dtype in train_data.dtypes if 'vector' in dtype]
non_vector_cols = [col_name for col_name in feature_cols if col_name not in vector_cols]

# Step 2: Function to apply VectorSizeHint and return size hint stages for the pipeline
def get_vector_size_hint_stage(data, col_name):
    # Sample a small portion of the data to determine vector size
    sample_fraction = 0.0001  # Using 0.01% of the data for sampling
    sampled_df = data.select(col_name).sample(False, sample_fraction).limit(1)
    sample_row = sampled_df.take(1)

    if sample_row:
        vector_size = len(sample_row[0][col_name])
        print(f"Column '{col_name}' vector size: {vector_size}")
        # Return a VectorSizeHint stage for the pipeline if vector size is valid
        if vector_size > 0:
            return VectorSizeHint(inputCol=col_name, size=vector_size)
    else:
        print(f"No sample found for column '{col_name}'. Skipping VectorSizeHint.")
    
    return None

# Step 3: Create a list of VectorSizeHint stages for each vector column
vector_size_hint_stages = []
for col_name in vector_cols:
    print(f"Getting VectorSizeHint for vector column: '{col_name}'")
    size_hint_stage = get_vector_size_hint_stage(train_data, col_name)
    if size_hint_stage:
        vector_size_hint_stages.append(size_hint_stage)

# Step 4: Combine vector and non-vector columns into a single list for VectorAssembler
final_input_cols = vector_cols + non_vector_cols
print("Final input columns for feature assembly:", final_input_cols)

# Step 5: Assemble final features column using VectorAssembler
final_assembler = VectorAssembler(inputCols=final_input_cols, outputCol="features")

# Step 6: Initialize StandardScaler
standardScaler = StandardScaler(inputCol="features", outputCol="features_scaled")

# Step 7: Create a pipeline with VectorSizeHint stages, VectorAssembler, and StandardScaler
pipeline_stages = vector_size_hint_stages + [final_assembler, standardScaler]
pipeline_final = Pipeline(stages=pipeline_stages)

# Step 8: Fit the pipeline on train_data
model_final = pipeline_final.fit(train_data)
print("Pipeline fitting done.")

# Step 9: Transform train, valid, and test datasets using the fitted pipeline
train_data = model_final.transform(train_data)
valid_data = model_final.transform(valid_data)
test_data = model_final.transform(test_data)
print("Pipeline transformation done.")

Getting VectorSizeHint for vector column: 'gender_onehot'
Column 'gender_onehot' vector size: 4
Getting VectorSizeHint for vector column: 'race_onehot'
Column 'race_onehot' vector size: 7
Final input columns for feature assembly: ['gender_onehot', 'race_onehot', 'age_of_TBI_diagnosis', 'MedicalHistory', 'Z21', 'M19', 'S68', 'Y30', 'B05', 'A23', 'H82', 'V89', 'I31', 'V72', 'R16', 'Q61', 'O12', 'X76', 'Z12', 'S39', 'L65', 'F25', 'G12', 'E02', 'X04', 'B79', 'F32', 'M54', 'B34', 'S60', 'Z19', 'T36', 'E44', 'E83', 'Q65', 'R71', 'D66', 'Z64', 'J60', 'D81', 'F21', 'Q14', 'C22', 'P00', 'B01', 'Z49', 'V24', 'R13', 'I44', 'O68', 'L74', 'D28', 'E56', 'H59', 'I01', 'R06', 'O14', 'K56', 'C78', 'R37', 'A46', 'Y02', 'P72', 'K62', 'S00', 'M92', 'K40', 'M46', 'R80', 'C95', 'K03', 'C24', 'J63', 'X13', 'P13', 'H51', 'I21', 'C77', 'Z08', 'D16', 'O11', 'I06', 'F99', 'J93', 'F01', 'A92', 'P96', 'H34', 'E35', 'Q62', 'W42', 'Z91', 'G96', 'I63', 'B39', 'Q80', 'W58', 'I77', 'R47', 'N99', 'L27', 'J81', 'D21', 'B

Pipeline fitting done.
Pipeline transformation done.


In [8]:
#Step7
train_data = train_data.withColumn('label',train_data.label.cast('double'))
valid_data = valid_data.withColumn('label',valid_data.label.cast('double'))
test_data = test_data.withColumn('label',test_data.label.cast('double'))

In [9]:
#Step8
from pyspark.sql.functions import col

# Count the number of 0's and 1's in the label column
count_zeros = train_data.filter(col('label') == 0).count()
count_ones = train_data.filter(col('label') == 1).count()

# Display the counts
print(f"Number of 0's in the label column: {count_zeros}")
print(f"Number of 1's in the label column: {count_ones}")

Number of 0's in the label column: 70345
Number of 1's in the label column: 10738


In [10]:
#Step9
from pyspark.sql.functions import col

# Number of 0's and 1's in the label column
count_zeros = 70345
count_ones = 10738

# Separate the majority and minority classes
majority_class_df = train_data.filter(col('label') == 0)
minority_class_df = train_data.filter(col('label') == 1)

# Upsampling
# Calculate the number of times we need to duplicate the minority class to match the desired count
upsample_ratio = int((count_zeros - count_ones) / count_ones)
remaining_minority_samples = (count_zeros - count_ones) % count_ones
# Duplicate the minority class DataFrame
upsampled_minority_class_df = minority_class_df
for i in range(upsample_ratio):
    upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df)
# Add remaining samples to reach the exact count
upsampled_minority_class_df = upsampled_minority_class_df.union(minority_class_df.sample(withReplacement=True, fraction=(remaining_minority_samples / count_ones)))

# Downsampling
# Calculate the fraction for downsampling the majority class
downsample_fraction = count_ones / count_zeros
# Sample the majority class to match the number of minority class samples
downsampled_majority_class_df = majority_class_df.sample(withReplacement=False, fraction=downsample_fraction)

# Combine the upsampled minority class with the downsampled majority class
train_data_balanced= upsampled_minority_class_df.union(downsampled_majority_class_df)

# Display the counts after balancing
print("Number of 0's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 0).count())
print("Number of 1's in the balanced DataFrame: ", train_data_balanced.filter(col('label') == 1).count())

Number of 0's in the balanced DataFrame:  10675
Number of 1's in the balanced DataFrame:  70388


In [12]:
#Step10
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier

# Step 6: Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Step 7: Function to perform cross-validation using train and validation sets
def manual_cross_validation(train_df: DataFrame, val_df: DataFrame, param_grid: dict, label_col: str):
    results = []
    best_model = None
    best_auc = 0.0  # Track the best AUC

    for max_depth in param_grid['maxDepth']:
        for max_iter in param_grid['maxIter']:
            # Train GBT model on the train data
            gbt = GBTClassifier(featuresCol="features", labelCol=label_col,
                                maxDepth=max_depth, maxIter=max_iter)

            model = gbt.fit(train_data_balanced)

            # Validate on validation set
            val_data_pred = model.transform(valid_data)

            # Calculate AUC using the manual function
            val_auc = calculate_manual_auc(val_data_pred, label_col, "probability")

            # Calculate accuracy for logging
            val_accuracy = calculate_accuracy(val_data_pred, label_col, "prediction")

            # Store results
            results.append((max_depth, max_iter, val_accuracy, val_auc))

            # Keep track of the best model based on validation AUC
            if val_auc > best_auc:
                best_auc = val_auc
                best_model = (gbt, model, val_accuracy, val_auc)

    return results, best_model[1]  # Return results and best model

def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    # Cast label and prediction to ensure they are the same type
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))

    # Compute correct predictions and total predictions
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)  # Round accuracy to 4 decimal places

# Function to calculate AUC manually with improved precision
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns (label and the probability for the positive class)
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Assuming the second column is the positive class probability

    # Sort predictions by probability, descending
    preds = preds.sortBy(lambda x: -x[1]).collect()

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    # Initialize variables for AUC calculation
    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Add precision control by avoiding division by zero
    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative

        # Trapezoidal area for AUC calculation
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)  # Round AUC to 4 decimal places

# Step 9: Run manual cross-validation on the balanced train and validation data
cv_results, best_model = manual_cross_validation(train_data_balanced, valid_data, param_grid, 'label')

# Find the best hyperparameters based on validation AUC
best_params = max(cv_results, key=lambda x: x[3])
print(f"Best parameters: maxDepth={best_params[0]}, maxIter={best_params[1]}, Validation AUC={best_params[3]}")

# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = best_model.transform(train_data_balanced)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")

# Transform validation data
val_data_pred = best_model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")

# Transform test data
test_data_pred = best_model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")

# Print results
print(f"Train Accuracy: {train_accuracy}, Train AUC: {train_auc}")
print(f"Validation Accuracy: {val_accuracy}, Validation AUC: {val_auc}")
print(f"Test Accuracy: {test_accuracy}, Test AUC: {test_auc}")

Best parameters: maxDepth=10, maxIter=20, Validation AUC=0.8685
Train Accuracy: 0.9113, Train AUC: 0.9349
Validation Accuracy: 0.5145, Validation AUC: 0.8685
Test Accuracy: 0.5177, Test AUC: 0.8647


In [13]:
#Step11
###################To align with MulticlassMetrics, swap the calculations for precision and recall between class 0 and class 1 in the code.##
################################precision_0 and recall_0 now represent the values that would be calculated for class 1###############
#############################precision_1 and recall_1 now represent the values that would be calculated for class 0#####################
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def calculate_confusion_matrix_metrics(df: DataFrame, label_col: str, prediction_col: str):
    # Calculate confusion matrix counts
    confusion_counts = df.groupBy(label_col, prediction_col).agg(F.count("*").alias("count"))

    # Initialize metrics as DataFrames
    true_positives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_positives"))
    true_negatives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_negatives"))
    false_positives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_positives"))
    false_negatives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_negatives"))

    # Combine all metrics into one DataFrame
    metrics = true_positives.crossJoin(true_negatives).crossJoin(false_positives).crossJoin(false_negatives)

    # Calculate overall counts
    total_count = df.count()
    accuracy = (metrics.select("true_positives").first()[0] + metrics.select("true_negatives").first()[0]) / total_count if total_count > 0 else 0.0

    # Extract metric values from the DataFrame
    tp = metrics.select("true_positives").first()[0]
    tn = metrics.select("true_negatives").first()[0]
    fp = metrics.select("false_positives").first()[0]
    fn = metrics.select("false_negatives").first()[0]

    # Calculate precision, recall, F1 score for class 0 (now swapping with class 1)
    precision_0 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_0 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    # Calculate precision, recall, F1 score for class 1 (now swapping with class 0)
    precision_1 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_1 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0.0

    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Return all metrics in a structured format
    return {
        "metrics": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        },
        "accuracy": accuracy,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "sensitivity": recall_1,  # Sensitivity is recall for class 1
        "f1_score_1": f1_score_1,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_score_0": f1_score_0,
        "specificity": specificity
    }

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
print("Confusion Matrix Metrics:")
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

# Print sensitivity separately
print(f"Sensitivity (Recall for Class 1): {metrics['sensitivity']:.4f}")

Confusion Matrix Metrics:
True Positives: 1480
True Negatives: 4625
False Positives: 5602
False Negatives: 85
Accuracy: 0.5177
Precision 1: 0.9820
Recall 1: 0.4522
Sensitivity: 0.4522
F1 Score 1: 0.6193
Precision 0: 0.2090
Recall 0: 0.9457
F1 Score 0: 0.3423
Specificity: 0.4522
Sensitivity (Recall for Class 1): 0.4522


In [11]:
#Extras to experiment ##############
######################Print Predictions to compare #################################################
from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier

# Step 6: Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Step 7: Function to perform cross-validation using train and validation sets
def manual_cross_validation(train_df: DataFrame, val_df: DataFrame, param_grid: dict, label_col: str):
    results = []
    best_model = None
    best_auc = 0.0  # Track the best AUC

    for max_depth in param_grid['maxDepth']:
        for max_iter in param_grid['maxIter']:
            # Train GBT model on the train data
            gbt = GBTClassifier(featuresCol="features", labelCol=label_col,
                                maxDepth=max_depth, maxIter=max_iter)

            model = gbt.fit(train_df)

            # Validate on validation set
            val_data_pred = model.transform(val_df)

            # Calculate AUC using the manual function
            val_auc = calculate_manual_auc(val_data_pred, label_col, "probability")

            # Calculate accuracy for logging
            val_accuracy = calculate_accuracy(val_data_pred, label_col, "prediction")

            # Store results
            results.append((max_depth, max_iter, val_accuracy, val_auc))

            # Print predicted probabilities and actual labels for validation data
            print("Validation Predicted Probabilities and Actual Labels:")
            val_data_pred.select("label", "probability").show(truncate=False)

            # Keep track of the best model based on validation AUC
            if val_auc > best_auc:
                best_auc = val_auc
                best_model = (gbt, model, val_accuracy, val_auc)

    return results, best_model[1]  # Return results and best model

def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    # Cast label and prediction to ensure they are the same type
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))

    # Compute correct predictions and total predictions
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)  # Round accuracy to 4 decimal places

# Function to calculate AUC manually with improved precision
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    # Collect the relevant columns (label and the probability for the positive class)
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1])))  # Assuming the second column is the positive class probability

    # Sort predictions by probability, descending
    preds = preds.sortBy(lambda x: -x[1]).collect()

    # Calculate True Positive Rate (TPR) and False Positive Rate (FPR)
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive

    # Initialize variables for AUC calculation
    tpr = 0.0
    fpr = 0.0
    auc = 0.0
    prev_fpr = 0.0
    prev_tpr = 0.0

    # Add precision control by avoiding division by zero
    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative

        # Trapezoidal area for AUC calculation
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)  # Round AUC to 4 decimal places

# Step 9: Run manual cross-validation on the balanced train and validation data
cv_results, best_model = manual_cross_validation(train_data_balanced, valid_data, param_grid, 'label')

# Find the best hyperparameters based on validation AUC
best_params = max(cv_results, key=lambda x: x[3])
print(f"Best parameters: maxDepth={best_params[0]}, maxIter={best_params[1]}, Validation AUC={best_params[3]}")

# Step 10: Calculate and print Train Accuracy using the best model
train_data_pred = best_model.transform(train_data_balanced)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")

# Transform validation data
val_data_pred = best_model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")

# Transform test data
test_data_pred = best_model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")

# Print predicted probabilities and actual labels for the training and test datasets
print("Training Predicted Probabilities and Actual Labels:")
train_data_pred.select("label", "probability").show(truncate=False)

print("Test Predicted Probabilities and Actual Labels:")
test_data_pred.select("label", "probability").show(truncate=False)

# Print results
print(f"Train Accuracy: {train_accuracy}, Train AUC: {train_auc}")
print(f"Validation Accuracy: {val_accuracy}, Validation AUC: {val_auc}")
print(f"Test Accuracy: {test_accuracy}, Test AUC: {test_auc}")

Validation Predicted Probabilities and Actual Labels:
+-----+----------------------------------------+
|label|probability                             |
+-----+----------------------------------------+
|1.0  |[0.08972741282925838,0.9102725871707417]|
|1.0  |[0.2445270690623313,0.7554729309376687] |
|1.0  |[0.09957710740077255,0.9004228925992275]|
|1.0  |[0.2646156402502076,0.7353843597497924] |
|1.0  |[0.0803476880909508,0.9196523119090492] |
|1.0  |[0.12022824861670883,0.8797717513832912]|
|1.0  |[0.07656612023721095,0.923433879762789] |
|1.0  |[0.08289266929975922,0.9171073307002408]|
|1.0  |[0.07875785605791363,0.9212421439420864]|
|1.0  |[0.08050908184988916,0.9194909181501109]|
|1.0  |[0.257432637334542,0.742567362665458]   |
|1.0  |[0.07243567393378088,0.9275643260662191]|
|1.0  |[0.09475653345815571,0.9052434665418443]|
|1.0  |[0.07139447491866964,0.9286055250813303]|
|1.0  |[0.2445270690623313,0.7554729309376687] |
|1.0  |[0.11853880644382007,0.8814611935561799]|
|1.0  |[0.11550

In [10]:
######################Old Calculation where precision_0 and recall_0: Class 1 and precision_1 and recall_1: Class0 ###################
# Function to calculate and print confusion matrix metrics
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def calculate_confusion_matrix_metrics(df: DataFrame, label_col: str, prediction_col: str):
    # Calculate confusion matrix counts
    confusion_counts = df.groupBy(label_col, prediction_col).agg(F.count("*").alias("count"))

    # Initialize metrics as DataFrames
    true_positives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_positives"))
    true_negatives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_negatives"))
    false_positives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_positives"))
    false_negatives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_negatives"))

    # Combine all metrics into one DataFrame
    metrics = true_positives.crossJoin(true_negatives).crossJoin(false_positives).crossJoin(false_negatives)

    # Calculate overall counts
    total_count = df.count()
    accuracy = (metrics.select("true_positives").first()[0] + metrics.select("true_negatives").first()[0]) / total_count if total_count > 0 else 0.0

    # Extract metric values from the DataFrame
    tp = metrics.select("true_positives").first()[0]
    tn = metrics.select("true_negatives").first()[0]
    fp = metrics.select("false_positives").first()[0]
    fn = metrics.select("false_negatives").first()[0]

    # Calculate precision, recall, F1 score for class 1
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0.0

    # Calculate precision, recall, F1 score for class 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Return all metrics in a structured format
    return {
        "metrics": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        },
        "accuracy": accuracy,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "sensitivity": recall_1,  # Add sensitivity (recall for class 1)
        "f1_score_1": f1_score_1,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_score_0": f1_score_0,
        "specificity": specificity
    }

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
print("Confusion Matrix Metrics:")
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

# Print sensitivity separately
print(f"Sensitivity (Recall for Class 1): {metrics['sensitivity']:.4f}")

Confusion Matrix Metrics:
True Positives: 1471
True Negatives: 4235
False Positives: 5992
False Negatives: 94
Accuracy: 0.4839
Precision 1: 0.1971
Recall 1: 0.9399
Sensitivity: 0.9399
F1 Score 1: 0.3259
Precision 0: 0.9783
Recall 0: 0.4141
F1 Score 0: 0.5819
Specificity: 0.4141
Sensitivity (Recall for Class 1): 0.9399


In [11]:
######################################Removed Intermediate Rounding in AUC Calculation #########################################
##################################################Added Predefined AUC Calculation############################################

from pyspark.sql import DataFrame
from pyspark.sql import functions as F
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

# Define hyperparameter grid for GBTClassifier
param_grid = {
    "maxDepth": [5, 10],
    "maxIter": [10, 20]
}

# Function to perform cross-validation using train and validation sets
def manual_cross_validation(train_df: DataFrame, val_df: DataFrame, param_grid: dict, label_col: str):
    results = []
    best_model = None
    best_auc = 0.0  # Track the best AUC

    for max_depth in param_grid['maxDepth']:
        for max_iter in param_grid['maxIter']:
            # Train GBT model on the train data
            gbt = GBTClassifier(featuresCol="features", labelCol=label_col, maxDepth=max_depth, maxIter=max_iter)
            model = gbt.fit(train_df)

            # Validate on validation set
            val_data_pred = model.transform(val_df)

            # Calculate AUC using the manual function
            val_auc = calculate_manual_auc(val_data_pred, label_col, "probability")

            # Calculate predefined AUC as baseline
            evaluator = BinaryClassificationEvaluator(labelCol=label_col, rawPredictionCol="probability", metricName="areaUnderROC")
            predefined_val_auc = evaluator.evaluate(val_data_pred)

            # Calculate accuracy for logging
            val_accuracy = calculate_accuracy(val_data_pred, label_col, "prediction")

            # Store results
            results.append((max_depth, max_iter, val_accuracy, val_auc, predefined_val_auc))

            # Print predicted probabilities and actual labels for validation data
            print("Validation Predicted Probabilities and Actual Labels:")
            val_data_pred.select("label", "probability").show(truncate=False)

            # Keep track of the best model based on validation AUC
            if val_auc > best_auc:
                best_auc = val_auc
                best_model = (gbt, model, val_accuracy, val_auc)

    return results, best_model[1]  # Return results and best model

# Accuracy calculation function
def calculate_accuracy(predictions: DataFrame, label_col: str, prediction_col: str) -> float:
    predictions = predictions.withColumn(label_col, F.col(label_col).cast('double'))
    predictions = predictions.withColumn(prediction_col, F.col(prediction_col).cast('double'))
    correct_predictions = predictions.filter(F.col(label_col) == F.col(prediction_col)).count()
    total_predictions = predictions.count()
    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0.0
    return round(accuracy, 4)

# AUC calculation function with only final rounding
def calculate_manual_auc(predictions: DataFrame, label_col: str, probability_col: str) -> float:
    preds = predictions.select(label_col, probability_col).rdd
    preds = preds.map(lambda row: (float(row[0]), float(row[1][1]))).sortBy(lambda x: -x[1]).collect()
    total_positive = sum(1 for label, _ in preds if label == 1.0)
    total_negative = len(preds) - total_positive
    tpr = fpr = auc = prev_fpr = prev_tpr = 0.0

    if total_positive == 0 or total_negative == 0:
        return 0.0

    for label, prob in preds:
        if label == 1.0:
            tpr += 1 / total_positive
        else:
            fpr += 1 / total_negative
        auc += (fpr - prev_fpr) * (tpr + prev_tpr) / 2
        prev_fpr = fpr
        prev_tpr = tpr

    return round(auc, 4)

# Run manual cross-validation on the balanced train and validation data
cv_results, best_model = manual_cross_validation(train_data_balanced, valid_data, param_grid, 'label')

# Find best hyperparameters based on validation AUC
best_params = max(cv_results, key=lambda x: x[3])
print(f"Best parameters: maxDepth={best_params[0]}, maxIter={best_params[1]}, Validation AUC={best_params[3]}")

# Calculate and print Train Accuracy using the best model
train_data_pred = best_model.transform(train_data_balanced)
train_accuracy = calculate_accuracy(train_data_pred, "label", "prediction")
train_auc = calculate_manual_auc(train_data_pred, "label", "probability")

# Calculate predefined train AUC as baseline
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="probability", metricName="areaUnderROC")
predefined_train_auc = evaluator.evaluate(train_data_pred)

# Transform validation data
val_data_pred = best_model.transform(valid_data)
val_accuracy = calculate_accuracy(val_data_pred, "label", "prediction")
val_auc = calculate_manual_auc(val_data_pred, "label", "probability")
predefined_val_auc = evaluator.evaluate(val_data_pred)

# Transform test data
test_data_pred = best_model.transform(test_data)
test_accuracy = calculate_accuracy(test_data_pred, "label", "prediction")
test_auc = calculate_manual_auc(test_data_pred, "label", "probability")
predefined_test_auc = evaluator.evaluate(test_data_pred)

# Print predicted probabilities and actual labels for the training and test datasets
print("Training Predicted Probabilities and Actual Labels:")
train_data_pred.select("label", "probability").show(truncate=False)

print("Test Predicted Probabilities and Actual Labels:")
test_data_pred.select("label", "probability").show(truncate=False)

# Print results
print(f"Train Accuracy: {train_accuracy}, Manual Train AUC: {train_auc}, Predefined Train AUC: {predefined_train_auc}")
print(f"Validation Accuracy: {val_accuracy}, Manual Validation AUC: {val_auc}, Predefined Validation AUC: {predefined_val_auc}")
print(f"Test Accuracy: {test_accuracy}, Manual Test AUC: {test_auc}, Predefined Test AUC: {predefined_test_auc}")

Validation Predicted Probabilities and Actual Labels:
+-----+----------------------------------------+
|label|probability                             |
+-----+----------------------------------------+
|1.0  |[0.1008565252206142,0.8991434747793858] |
|1.0  |[0.23976001059578136,0.7602399894042187]|
|1.0  |[0.08332100797572033,0.9166789920242797]|
|1.0  |[0.2821240645462375,0.7178759354537625] |
|1.0  |[0.07220394268135318,0.9277960573186468]|
|1.0  |[0.14923033674459465,0.8507696632554054]|
|1.0  |[0.07818066175592302,0.921819338244077] |
|1.0  |[0.08334874937089884,0.9166512506291011]|
|1.0  |[0.06715782569217928,0.9328421743078207]|
|1.0  |[0.08702477966463625,0.9129752203353637]|
|1.0  |[0.23976001059578136,0.7602399894042187]|
|1.0  |[0.07034700185697011,0.9296529981430299]|
|1.0  |[0.07509257860801737,0.9249074213919827]|
|1.0  |[0.0787315779035215,0.9212684220964785] |
|1.0  |[0.23976001059578136,0.7602399894042187]|
|1.0  |[0.14757208735645028,0.8524279126435497]|
|1.0  |[0.07537

In [12]:
######################Old Calculation where precision_0 and recall_0: Class 1 and precision_1 and recall_1: Class0 ###################
# Function to calculate and print confusion matrix metrics
from pyspark.sql import DataFrame
import pyspark.sql.functions as F

def calculate_confusion_matrix_metrics(df: DataFrame, label_col: str, prediction_col: str):
    # Calculate confusion matrix counts
    confusion_counts = df.groupBy(label_col, prediction_col).agg(F.count("*").alias("count"))

    # Initialize metrics as DataFrames
    true_positives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_positives"))
    true_negatives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("true_negatives"))
    false_positives = confusion_counts.filter((F.col(label_col) == 0) & (F.col(prediction_col) == 1)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_positives"))
    false_negatives = confusion_counts.filter((F.col(label_col) == 1) & (F.col(prediction_col) == 0)).select(F.coalesce(F.first("count"), F.lit(0)).alias("false_negatives"))

    # Combine all metrics into one DataFrame
    metrics = true_positives.crossJoin(true_negatives).crossJoin(false_positives).crossJoin(false_negatives)

    # Calculate overall counts
    total_count = df.count()
    accuracy = (metrics.select("true_positives").first()[0] + metrics.select("true_negatives").first()[0]) / total_count if total_count > 0 else 0.0

    # Extract metric values from the DataFrame
    tp = metrics.select("true_positives").first()[0]
    tn = metrics.select("true_negatives").first()[0]
    fp = metrics.select("false_positives").first()[0]
    fn = metrics.select("false_negatives").first()[0]

    # Calculate precision, recall, F1 score for class 1
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1_score_1 = (2 * precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0.0

    # Calculate precision, recall, F1 score for class 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0.0
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    f1_score_0 = (2 * precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0.0

    # Specificity
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

    # Return all metrics in a structured format
    return {
        "metrics": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        },
        "accuracy": accuracy,
        "precision_1": precision_1,
        "recall_1": recall_1,
        "sensitivity": recall_1,  # Add sensitivity (recall for class 1)
        "f1_score_1": f1_score_1,
        "precision_0": precision_0,
        "recall_0": recall_0,
        "f1_score_0": f1_score_0,
        "specificity": specificity
    }

# Calculate and print metrics for test predictions
metrics = calculate_confusion_matrix_metrics(test_data_pred, "label", "prediction")

# Print results
print("Confusion Matrix Metrics:")
for metric_name, metric_value in metrics.items():
    if metric_name == "metrics":
        for key, value in metric_value.items():
            print(f"{key.replace('_', ' ').title()}: {value}")
    else:
        print(f"{metric_name.replace('_', ' ').title()}: {metric_value:.4f}")

# Print sensitivity separately
print(f"Sensitivity (Recall for Class 1): {metrics['sensitivity']:.4f}")

Confusion Matrix Metrics:
True Positives: 1561
True Negatives: 35
False Positives: 10192
False Negatives: 4
Accuracy: 0.1353
Precision 1: 0.1328
Recall 1: 0.9974
Sensitivity: 0.9974
F1 Score 1: 0.2344
Precision 0: 0.8974
Recall 0: 0.0034
F1 Score 0: 0.0068
Specificity: 0.0034
Sensitivity (Recall for Class 1): 0.9974
